In [ ]:
import pandas as pd
import numpy as np

from math import ceil

#import matplotlib.pyplot as plt
import sys, os, time, random, re, csv, json, argparse, torch#,copy , indexer, evaluate

from glob import glob
from datetime import datetime
from datasets import Dataset, Value, concatenate_datasets 
from sklearn.metrics import mean_squared_error, f1_score, accuracy_score, precision_score, recall_score, classification_report
from scipy.special import expit

from torch.nn import functional as F#, BCEWithLogitsLoss
from torch.utils.data import WeightedRandomSampler

from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, 
                          BertForSequenceClassification, BertModel, EarlyStoppingCallback, AdamW,
                          PreTrainedModel, Trainer, TrainingArguments, get_scheduler)

sys.path.append("/Proyecto/Value-disagreement/Python/Utilities")
import Dict_Object #, text_cleansing

In [ ]:
def get_dataset(table):
    # --- ### VALUENET    
    if table == "valueNET":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test.csv",sep=',')

        value_train_set = np.loadtxt("/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test.csv", delimiter=',', dtype=int)

    # --- ### VALUEARG
    elif table == "valueARG":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test.csv",sep=',')

        value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test.csv", delimiter=',', dtype=int)

    # --- ### All (valueNET+valueARG)
    elif table == "valueALL":
        value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all.csv", sep='|')

        #value_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv",sep=',')
        #value_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv",sep=',')
        #value_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv",sep=',')
    
        value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv", delimiter=',', dtype=int)
        value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv", delimiter=',', dtype=int)
        value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv", delimiter=',', dtype=int)

    return value_set, value_train_set, value_val_set, value_test_set

# cargar todo
value_set, value_train_set, value_val_set, value_test_set = get_dataset("valueALL")

#value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv", delimiter=',', dtype=int)
test_df = value_set.iloc[value_test_set].reset_index(drop=True)
test_df = test_df.rename(columns={"scenario": "body_cleand", "uid": "id"})
test_df["author"] = test_df["id"].astype(str)
test_df

In [ ]:
test_df["value"] = test_df["value"].str.lower()
test_df

In [ ]:
test_df[test_df["id"]=="A12428"]

In [ ]:
test_df.to_csv(r"/Proyecto/Value-disagreement/Python/RESULTADOS/test_all.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

In [ ]:
### DEFINIMOS UNA VARIABLE QUE NOS FACILITE LA RUTA DE LOS ARCHIVOS
program_path = "/Proyecto/Value-disagreement/Python/Models/Inference/deba_usr_comments"
print(program_path)

In [ ]:
def ObtenerArchivos(ruta_actual,carpeta):
    """Esta función nos devuelve una lista con los archivos de una carpeta"""
    ruta_completa = os.path.join(ruta_actual,carpeta)
    archivos = glob(ruta_completa+"/*")
    return archivos

In [ ]:
files = ObtenerArchivos(program_path,'TEST')
list(files)

In [ ]:
lista_txt = []
for archivo in files:
    print("archivo: ",archivo)
    df = pd.read_csv(archivo, sep="|")
    lista_txt.append(df)
        
test_inf = pd.concat(lista_txt,axis=0, sort = False)
test_inf.reset_index(inplace=True,drop=True)

In [ ]:
test_inf

In [ ]:
test_inf[test_inf["id"]=="A20137"]

In [ ]:
test_inf=test_inf.drop_duplicates(subset=["id","author","value","pred"])

In [ ]:
test_inf["key"] = test_inf["id"].str.strip() + "-" + test_inf["value"].str.strip()
test_inf

In [ ]:
test_inf[test_inf["id"]=="A12428"]

In [ ]:
test_df["key"] = test_df["id"].astype(str).str.strip()  + "-" + test_df["value"].str.strip()
test_df#.dtypes

In [ ]:
test_df[test_df["key"]=='A20137-power']

In [ ]:
test_label = pd.merge(test_inf, test_df, how='left', left_on='key', right_on='key')
test_label

In [ ]:
test_label=test_label[test_label['label'].isnull() == False]

In [ ]:
test_label.label = test_label.label.fillna(0) 

In [ ]:
test_label.drop(columns=['id_y','body_cleand','value_y','author_y'], inplace=True)
test_label

In [ ]:
def safe_div(a, b):
    return a / b if b else np.nan

def prf_from_counts(tp, fp, fn):
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall) if (precision is not np.nan and recall is not np.nan and (precision + recall)) else np.nan
    return precision, recall, f1

def metrics_for_group(g: pd.DataFrame, pred_col: str):
    y = g["label"].astype(int).to_numpy()
    p = g[pred_col].astype(int).to_numpy()

    tp = int(((p == 1) & (y == 1)).sum())
    fp = int(((p == 1) & (y == 0)).sum())
    fn = int(((p == 0) & (y == 1)).sum())
    tn = int(((p == 0) & (y == 0)).sum())

    precision, recall, f1 = prf_from_counts(tp, fp, fn)
    support_pos = int((y == 1).sum())
    n = int(len(g))
    pos_rate = float((p == 1).mean())

    return pd.Series({
        "n": n,
        "support_pos": support_pos,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pos_rate": pos_rate,
    })

In [ ]:
df = test_label.copy()
df["label"] = df["label"].astype(int)
df["pred"] = df["pred"].astype(int)
df["pred_05"] = (df["prob"].astype(float) >= 0.5).astype(int)

In [ ]:
# Tabla por valor: 0.5 vs calibrado

by_value_05 = df.groupby("value_x").apply(metrics_for_group, pred_col="pred_05").reset_index()
by_value_cal = df.groupby("value_x").apply(metrics_for_group, pred_col="pred").reset_index()

rename_map = {
    "precision": "precision_05",
    "recall": "recall_05",
    "f1": "f1_05",
    "pos_rate": "pos_rate_05",
    "tp": "tp_05", "fp": "fp_05", "fn": "fn_05", "tn": "tn_05"
}
by_value_05 = by_value_05.rename(columns=rename_map)

rename_map = {
    "precision": "precision_cal",
    "recall": "recall_cal",
    "f1": "f1_cal",
    "pos_rate": "pos_rate_cal",
    "tp": "tp_cal", "fp": "fp_cal", "fn": "fn_cal", "tn": "tn_cal"
}
by_value_cal = by_value_cal.rename(columns=rename_map)

# Unir (n y support_pos deberían ser iguales entre ambos, por valor)
t_impact = by_value_05.merge(
    by_value_cal,
    on=["value_x", "n", "support_pos"],
    how="inner"
)

# Deltas
t_impact["delta_f1"] = t_impact["f1_cal"] - t_impact["f1_05"]
t_impact["delta_precision"] = t_impact["precision_cal"] - t_impact["precision_05"]
t_impact["delta_recall"] = t_impact["recall_cal"] - t_impact["recall_05"]
t_impact["delta_pos_rate"] = t_impact["pos_rate_cal"] - t_impact["pos_rate_05"]

# Ordenar (opcional)
t_impact = t_impact.sort_values("value_x").reset_index(drop=True)

# Selección de columnas “tesis-friendly”
table_52 = t_impact[[
    "value_x", "n", "support_pos",
    "precision_05", "recall_05", "f1_05", "pos_rate_05",
    "precision_cal", "recall_cal", "f1_cal", "pos_rate_cal",
    "delta_precision", "delta_recall", "delta_f1", "delta_pos_rate"
]].copy()

# Formato redondeado para imprimir/pegar (no modifica valores originales si querés guardar sin redondeo)
table_52_round = table_52.copy()
num_cols = [c for c in table_52_round.columns if c not in ("value_x", "n", "support_pos")]
table_52_round[num_cols] = table_52_round[num_cols].round(4)

print("\n=== Tabla 5.2 (por valor): Impacto calibración (0.5 vs calibrado) ===")
table_52_round

In [ ]:
table_52_round.f1_05.mean()

In [ ]:
table_52_round.f1_cal.mean()

In [ ]:
table_52_round[["value_x","n","support_pos","precision_05","recall_05","f1_05","pos_rate_05","precision_cal","recall_cal","f1_cal","pos_rate_cal"]]

In [ ]:
df_values = test_label.copy()

y_true = df_values["label"].astype(int)
y_pred = df_values["pred"].astype(int)

micro_precision = precision_score(y_true, y_pred, average="micro")
micro_recall    = recall_score(y_true, y_pred, average="micro")
micro_f1        = f1_score(y_true, y_pred, average="micro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall    = recall_score(y_true, y_pred, average="macro")
macro_f1        = f1_score(y_true, y_pred, average="macro")

print("Micro Precision:", round(micro_precision,4))
print("Micro Recall:", round(micro_recall,4))
print("Micro F1:", round(micro_f1,4))

print("\nMacro Precision:", round(macro_precision,4))
print("Macro Recall:", round(macro_recall,4))
print("Macro F1:", round(macro_f1,4))

In [ ]:
# 3) Tabla resumen global (macro promedio sobre valores)
summary_rows = []

for name, pref, rec, f1, pr in [
    ("Umbral 0.5", "precision_05", "recall_05", "f1_05", "pos_rate_05"),
    ("Calibrado",  "precision_cal","recall_cal","f1_cal","pos_rate_cal"),
]:
    summary_rows.append({
        "setting": name,
        "macro_precision": float(table_52[pref].mean(skipna=True)),
        "macro_recall": float(table_52[rec].mean(skipna=True)),
        "macro_f1": float(table_52[f1].mean(skipna=True)),
        "avg_pos_rate": float(table_52[pr].mean(skipna=True)),
    })

table_52_macro = pd.DataFrame(summary_rows)
table_52_macro_round = table_52_macro.copy()
for c in ["macro_precision","macro_recall","macro_f1","avg_pos_rate"]:
    table_52_macro_round[c] = table_52_macro_round[c].round(4)

print("\n=== Tabla resumen (macro sobre valores) ===")
table_52_macro_round

In [ ]:
# Tabla Positive Rate

table_pos_rate = table_52[[
    "value_x", "pos_rate_05", "pos_rate_cal", "delta_pos_rate"
]].copy()
table_pos_rate_round = table_pos_rate.copy()
table_pos_rate_round[["pos_rate_05","pos_rate_cal","delta_pos_rate"]] = table_pos_rate_round[["pos_rate_05","pos_rate_cal","delta_pos_rate"]].round(4)

print("\n=== Tabla (opcional): Distribución de activaciones (% positivos) ===")
table_pos_rate_round

In [ ]:
# table_52_round.to_csv("tabla_5_2_impacto_calibracion.csv", index=False)
# table_52_macro_round.to_csv("tabla_5_2_resumen_macro.csv", index=False)
# table_pos_rate_round.to_csv("tabla_activaciones_por_valor.csv", index=False)

In [ ]:
### F! by value

program_path = "/Proyecto/Value-disagreement/Python/Results/Per Value/Para resultados tesis"
#program_path = "/Proyecto/Value-disagreement/Python/Results/Per Value/Para resultados tesis
files = ObtenerArchivos(program_path,'infer')
#files = ObtenerArchivos(program_path,'eval')
list(files)

In [ ]:
rows = []
for path in files:
    # ejemplo: bert-base-uncased_valueALL_seed0_per_value_f1.json
    fname = path.split("/")[-1]

    model = fname.split("_valueALL")[0]
    seed = int(fname.split("seed")[1].split("_")[0])

    with open(path, "r") as f:
        data = json.load(f)

    for value, f1 in data.items():
        rows.append({"model": model,
                     "seed": seed,
                     "value": value,
                     "f1": f1
                     })

df = pd.DataFrame(rows)
df

In [ ]:
df[df['value']=="ACHIEVEMENT"]

In [ ]:
agg = (df.groupby(["value", "model"])["f1"].agg(["mean", "std"]).reset_index())
agg

In [ ]:
agg2 = (df.groupby(["model"])["f1"].agg(["mean", "std"]).reset_index())
agg2

In [ ]:
table = agg.pivot(index="value", columns="model", values="mean")
table_std = agg.pivot(index="value", columns="model", values="std")

aux = table.round(3).astype(str) + " ± " + table_std.round(3).astype(str)
aux = aux.reset_index()

best_model = table.idxmax(axis=1).reset_index()
best_model.columns = ["value", "best_model"]

final = aux.merge(best_model, on="value")
final

In [ ]:
final

In [ ]:
final["roberta-base"]

In [ ]:
# HEATMAP VALORES
df = test_label.copy()

df["label"] = df["label"].astype(int)
df["pred"] = df["pred"].astype(int)

# Componentes de confusión
df["TP"] = ((df["label"] == 1) & (df["pred"] == 1)).astype(int)
df["FP"] = ((df["label"] == 0) & (df["pred"] == 1)).astype(int)
df["FN"] = ((df["label"] == 1) & (df["pred"] == 0)).astype(int)
df["TN"] = ((df["label"] == 0) & (df["pred"] == 0)).astype(int)

g = df.groupby("value_x")[["TP","FP","FN","TN"]].sum()

g["Recall"] = g["TP"] / (g["TP"] + g["FN"])
g["Precision"] = g["TP"] / (g["TP"] + g["FP"])
g["FNR"] = g["FN"] / (g["FN"] + g["TP"])
g["FPR"] = g["FP"] / (g["FP"] + g["TN"])

g = g.sort_index()

print(g[["Recall","Precision","FNR","FPR"]])

In [ ]:
import matplotlib.pyplot as plt

# Asegurar orden coherente (ajusta si usas otro orden)
order_vals = [
    "universalism", "benevolence", "conformity", "security", "power",
    "achievement", "stimulation", "tradition", "self-direction", "hedonism"
]

g_plot = g.loc[order_vals]

# Matriz 10x2: columnas = [FNR, FPR]
data = g_plot[["FNR", "FPR"]].to_numpy()

fig, ax = plt.subplots(figsize=(6, 0.45 * len(g_plot) + 2.5))

im = ax.imshow(data, aspect="auto")

# Ejes
ax.set_xticks([0, 1])
ax.set_xticklabels(["FNR", "FPR"])
ax.set_yticks(np.arange(len(g_plot)))
ax.set_yticklabels(g_plot.index)

ax.set_title("Error rates por valor\n(FNR = falsos negativos, FPR = falsos positivos)")
ax.set_ylabel("Valor")

# Barra de color
cbar = fig.colorbar(im, ax=ax)
cbar.ax.set_ylabel("Rate", rotation=90)

# Anotar valores en celdas
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center")

plt.tight_layout()

# Exportar
plt.savefig("heatmap_error_rates.png", dpi=300, bbox_inches="tight")
plt.savefig("heatmap_error_rates.pdf", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import ast

# --------------------
# Config
# --------------------
PATH = "/Proyecto/Value-disagreement/Python/Models/Inference/final_profiles/deba_with_profiles.csv"   # ajusta si es necesario
L = 100                           # ℓ (robustez)
N_SAMPLES = 5
RANDOM_STATE = 42

df = pd.read_csv(PATH, sep="|")

def parse_vec(x):
    return np.array(ast.literal_eval(x), dtype=float)

# Distancia euclídea 10D entre perfiles probabilísticos
p_mat = np.vstack(df["p_prob_vector"].apply(parse_vec).to_numpy())
c_mat = np.vstack(df["c_prob_vector"].apply(parse_vec).to_numpy())
df["dist_10d"] = np.linalg.norm(p_mat - c_mat, axis=1)

# Filtros: A (sin neutrales) + Cℓ
filtered = df[
    (df["label"].isin([0, 2])) &
    (df["p_total_value_mentions"] >= L) &
    (df["c_total_value_mentions"] >= L)
].copy()

# Percentiles globales de distancia (P95 y P5)
p95 = np.percentile(filtered["dist_10d"], 95)
p5  = np.percentile(filtered["dist_10d"], 5)

high_dist = filtered[filtered["dist_10d"] >= p95]
low_dist  = filtered[filtered["dist_10d"] <= p5]

# Conteos por label dentro de cada pool
# label: 2=agree, 0=disagree
counts = {
    "High distance + Agree": int((high_dist["label"] == 2).sum()),
    "High distance + Disagree": int((high_dist["label"] == 0).sum()),
    "Low distance + Agree": int((low_dist["label"] == 2).sum()),
    "Low distance + Disagree": int((low_dist["label"] == 0).sum()),
}

# Proporciones dentro de cada pool (alto vs bajo)
props = {
    "High distance + Agree": counts["High distance + Agree"] / len(high_dist),
    "High distance + Disagree": counts["High distance + Disagree"] / len(high_dist),
    "Low distance + Agree": counts["Low distance + Agree"] / len(low_dist),
    "Low distance + Disagree": counts["Low distance + Disagree"] / len(low_dist),
}

print(f"Filtered N = {len(filtered)} | High pool N = {len(high_dist)} | Low pool N = {len(low_dist)}")
print(f"P95(dist) = {p95:.6f} | P5(dist) = {p5:.6f}\n")

print("=== COUNTS ===")
for k, v in counts.items():
    print(f"{k}: {v}")

print("\n=== PROPORTIONS (within pool) ===")
for k, v in props.items():
    print(f"{k}: {v:.4f}")

In [ ]:
# Ejemplos DesAcuerdo
import ast

# ---------- Helpers ----------
def parse_vec(x):
    return np.array(ast.literal_eval(x), dtype=float)

# ---------- Compute 10D distance ----------
p_mat = np.vstack(df["p_prob_vector"].apply(parse_vec).to_numpy())
c_mat = np.vstack(df["c_prob_vector"].apply(parse_vec).to_numpy())
df["dist_10d"] = np.linalg.norm(p_mat - c_mat, axis=1)

# ---------- Filters (A + Cℓ) ----------
filtered = df[
    (df["label"].isin([0, 2])) &
    (df["p_total_value_mentions"] >= L) &
    (df["c_total_value_mentions"] >= L)
].copy()

# ---------- Global percentiles over ALL filtered rows ----------
p95 = np.percentile(filtered["dist_10d"], 95)
p5  = np.percentile(filtered["dist_10d"], 5)

high_dist_pool = filtered[filtered["dist_10d"] >= p95].copy()
low_dist_pool  = filtered[filtered["dist_10d"] <= p5].copy()

# ---------- Split pools by label ----------
high_agree_pool    = high_dist_pool[high_dist_pool["label"] == 2]
high_disagree_pool = high_dist_pool[high_dist_pool["label"] == 0]

low_agree_pool     = low_dist_pool[low_dist_pool["label"] == 2]
low_disagree_pool  = low_dist_pool[low_dist_pool["label"] == 0]

# ---------- Sample examples (5 each) ----------
# NOTE: if any pool has < N_SAMPLES, sample will fail; adjust N_SAMPLES or use replace=True.
high_agree_examples = high_agree_pool.sample(N_SAMPLES, random_state=RANDOM_STATE)
high_disagree_examples = high_disagree_pool.sample(N_SAMPLES, random_state=RANDOM_STATE)
low_agree_examples = low_agree_pool.sample(N_SAMPLES, random_state=RANDOM_STATE)
low_disagree_examples = low_disagree_pool.sample(N_SAMPLES, random_state=RANDOM_STATE)

# ---------- Select columns to inspect ----------
cols = ["label", "subreddit", "dist_10d", "body_parent", "body_child",
        "p_total_value_mentions", "c_total_value_mentions"]

print("\n=== HIGH DISTANCE + AGREE (dist >= P95 global) ===")
print(f"P95 threshold (global): {p95:.6f} | pool size: {len(high_agree_pool)}")
print(high_agree_examples[cols].to_string(index=False))

print("\n=== HIGH DISTANCE + DISAGREE (dist >= P95 global) ===")
print(f"P95 threshold (global): {p95:.6f} | pool size: {len(high_disagree_pool)}")
print(high_disagree_examples[cols].to_string(index=False))

print("\n=== LOW DISTANCE + AGREE (dist <= P5 global) ===")
print(f"P5 threshold (global): {p5:.6f} | pool size: {len(low_agree_pool)}")
print(low_agree_examples[cols].to_string(index=False))

print("\n=== LOW DISTANCE + DISAGREE (dist <= P5 global) ===")
print(f"P5 threshold (global): {p5:.6f} | pool size: {len(low_disagree_pool)}")
print(low_disagree_examples[cols].to_string(index=False))

# ---------- Save to CSV ----------
out = pd.concat([
    high_agree_examples.assign(example_type="high_distance_agree"),
    high_disagree_examples.assign(example_type="high_distance_disagree"),
    low_agree_examples.assign(example_type="low_distance_agree"),
    low_disagree_examples.assign(example_type="low_distance_disagree"),
], ignore_index=True)

out_path = f"qualitative_examples_4types_L{L}_seed{RANDOM_STATE}.csv"
out.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")